<a href="https://colab.research.google.com/github/samyuktha07-prog/Genz-translator/blob/main/genz_translate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:


!pip -q install gradio pandas

import gradio as gr
import pandas as pd
import re
import random
from datetime import datetime


GENZ_MAP = {
    "hello": "yo",
    "hi": "heyy",
    "hey": "yoo",
    "yes": "yesss",
    "yes, definitely": "fr fr",
    "no": "nah",
    "okay": "bet",
    "ok": "bet",
    "okay then": "aight bet",
    "good": "valid",
    "great": "goated",
    "amazing": "insane",
    "very good": "goated",
    "beautiful": "gorgeous fr",
    "funny": "sending me 😭",
    "really": "literally",
    "very": "lowkey",
    "love": "luv",
    "friend": "bestie",
    "friends": "besties",
    "brother": "bro",
    "sister": "sis",
    "person": "bro",
    "people": "y'all",
    "everyone": "y'all",
    "understand": "get it",
    "understood": "gotchu",
    "I understand": "gotchu",
    "thank you": "tysm",
    "thanks": "ty",
    "please": "pls",
    "because": "bc",
    "before": "b4",
    "tonight": "tn",
    "tomorrow": "tmrw",
    "today": "today fr",
    "really good": "actually fire",
    "very funny": "hilarious af",
    "I am tired": "I'm cooked",
    "I am tired.": "I'm cooked 😭",
    "I am busy": "I'm booked and busy",
    "I don't know": "idk",
    "I do not know": "idk",
    "I don't care": "idc",
    "I cannot": "I can't 😭",
    "I can't": "nah I can't 😭",
    "wait": "hold up",
    "stop": "be so fr",
    "seriously": "be so fr",
    "that's true": "that's so real",
    "that is true": "that's so real",
    "I agree": "real",
    "exactly": "literally",
    "correct": "valid",
    "wrong": "L take",
    "bad": "not it",
    "terrible": "criminal 💀",
    "embarrassing": "that's actually crazy 😭",
    "embarrassed": "I'm cooked 😭",
    "confusing": "I'm lost bro",
    "confused": "I'm lost",
    "crazy": "unhinged",
    "shocking": "nahhh 💀",
    "surprising": "ain't no way",
    "amazing": "ate",
    "excellent": "ate and left no crumbs",
    "success": "W",
    "successful": "W",
    "failure": "L",
    "failed": "took an L",
    "problem": "an L",
    "situation": "the lore",
    "story": "the lore",
    "gossip": "tea",
    "rumor": "tea",
    "joke": "bit",
    "photo": "pic",
    "picture": "pic",
    "message": "text",
    "relationship": "situationship",
    "very attractive": "fine fr",
    "attractive": "lowkey fine",
    "awesome": "fire",
    "cool": "valid",
    "excited": "hyped",
    "happy": "we're so up",
    "sad": "I'm in my flop era",
    "angry": "mad pressed",
    "leave me alone": "let me breathe 😭",
    "I am leaving": "I'm out",
    "goodbye": "peace",
}


EMOJIS = {
    "happy": ["😭", "✨", "🫶", "💅", "🔥"],
    "excited": ["🔥", "😭", "🗣️", "‼️", "✨"],
    "sad": ["😭", "🥲", "💔", "🫠"],
    "angry": ["😭", "💀", "🙄", "🤨"],
    "confused": ["😭", "💀", "🤨", "🧍"],
    "funny": ["😭", "💀", "😂", "NAH"],
    "neutral": ["😭", "✨", "fr", "ngl"],
}


history = []

def clean_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


def detect_tone(text):

    t = text.lower()

    if any(x in t for x in [
        "love", "happy", "amazing", "great", "excited",
        "good", "awesome", "beautiful"
    ]):
        return "😍 Positive"

    if any(x in t for x in [
        "hate", "angry", "annoyed", "stupid", "bad",
        "terrible", "worst"
    ]):
        return "😤 Angry"

    if any(x in t for x in [
        "sad", "cry", "miss", "hurt", "lonely"
    ]):
        return "🥲 Sad"

    if any(x in t for x in [
        "why", "what", "confused", "understand"
    ]):
        return "🤨 Confused"

    return "😐 Neutral"


def calculate_genz_score(text):

    t = text.lower()

    slang_words = [
        "bro", "bestie", "fr", "ngl", "idk", "idc",
        "lowkey", "highkey", "bet", "sus", "slay",
        "ate", "goated", "fire", "valid", "real",
        "yap", "cooked", "lore", "tea", "rizz",
        "vibe", "cap", "no cap", "deadass"
    ]

    score = 0

    for word in slang_words:
        if word in t:
            score += 7

    if "!" in text:
        score += 5

    if "😭" in text or "💀" in text:
        score += 10

    if text.lower() != text and any(c.isupper() for c in text):
        score += 5

    return min(score, 100)


def translate_to_genz(text):

    original = clean_text(text)

    if not original:
        return ""

    result = original

    phrases = sorted(
        GENZ_MAP.items(),
        key=lambda x: len(x[0]),
        reverse=True
    )

    for normal, slang in phrases:

        pattern = r"\b" + re.escape(normal) + r"\b"

        result = re.sub(
            pattern,
            slang,
            result,
            flags=re.IGNORECASE
        )


    result = re.sub(
        r"\bvery\b",
        "lowkey",
        result,
        flags=re.IGNORECASE
    )


    lower = result.lower()

    if not any(
        x in lower
        for x in ["fr", "ngl", "lowkey", "bro", "bestie"]
    ):
        if random.random() > 0.5:
            result = result + " fr"

    return result

def translate_to_formal(text):

    result = clean_text(text)

    replacements = {
        "bro": "friend",
        "bestie": "friend",
        "y'all": "everyone",
        "fr": "truly",
        "ngl": "honestly",
        "idk": "I do not know",
        "idc": "I do not mind",
        "bet": "certainly",
        "lowkey": "somewhat",
        "highkey": "strongly",
        "goated": "excellent",
        "fire": "excellent",
        "ate": "performed excellently",
        "W": "success",
        "L": "failure",
        "cooked": "exhausted",
        "tea": "information",
        "lore": "background",
        "vibe": "atmosphere",
        "sus": "suspicious",
    }

    for slang, formal in replacements.items():

        result = re.sub(
            r"\b" + re.escape(slang) + r"\b",
            formal,
            result,
            flags=re.IGNORECASE
        )

    return result


def translate_funny(text):

    result = translate_to_genz(text)

    endings = [
        " 😭",
        " 💀",
        " bro be so fr",
        " 😭🙏",
        " literally me",
        " nahhh 💀",
        " and that's the tea ☕",
        " it's giving main character 😭"
    ]

    if not any(x in result for x in ["😭", "💀", "🙏", "☕"]):
        result += random.choice(endings)

    return result


def analyze(text, mode):

    if not text or not text.strip():

        return (
            "",
            "—",
            "—",
            "—",
            "—",
            "Enter some text first 👀"
        )

    original = clean_text(text)

    if mode == "Gen-Z Translation":
        output = translate_to_genz(original)

    elif mode == "Formal Translation":
        output = translate_to_formal(original)

    elif mode == "Funny Gen-Z":
        output = translate_funny(original)

    else:
        output = translate_to_genz(original)

    tone = detect_tone(original)

    score = calculate_genz_score(output)

    word_count = len(original.split())

    slang_detected = []

    combined = original.lower() + " " + output.lower()

    for slang in [
        "bro", "bestie", "fr", "ngl", "idk",
        "lowkey", "highkey", "bet", "ate",
        "goated", "fire", "valid", "cooked",
        "lore", "tea", "W", "L"
    ]:
        if slang.lower() in combined:
            slang_detected.append(slang)

    if slang_detected:
        slang_result = ", ".join(slang_detected)
    else:
        slang_result = "None detected"

    emoji_type = "happy"

    if "Sad" in tone:
        emoji_type = "sad"
    elif "Angry" in tone:
        emoji_type = "angry"
    elif "Confused" in tone:
        emoji_type = "confused"
    elif "Positive" in tone:
        emoji_type = "happy"

    emoji_result = " ".join(
        random.sample(
            EMOJIS[emoji_type],
            min(3, len(EMOJIS[emoji_type]))
        )
    )

    if score >= 70:
        verdict = "🔥 Certified Gen-Z. You ate."
    elif score >= 40:
        verdict = "👀 You're getting there. Very online."
    elif score >= 20:
        verdict = "😎 Some Gen-Z energy detected."
    else:
        verdict = "🧍 Bro is still living in 2012."

    history.append({
        "Time": datetime.now().strftime("%H:%M:%S"),
        "Mode": mode,
        "Gen-Z Score": score,
        "Tone": tone
    })

    return (
        output,
        str(score) + "/100",
        tone,
        slang_result,
        emoji_result,
        verdict
    )


def random_example():

    examples = [
        "Hello everyone, I am very excited about our project.",
        "I do not know what is happening right now.",
        "That was an amazing performance.",
        "Thank you for helping me with this.",
        "I am really tired after college today.",
        "This situation is very confusing.",
        "That is actually very funny.",
        "I completely agree with you.",
        "I cannot believe you actually did that!",
        "Today was a really good day."
    ]

    return random.choice(examples)


def show_history():

    if not history:
        return pd.DataFrame(
            columns=[
                "Time",
                "Mode",
                "Gen-Z Score",
                "Tone"
            ]
        )

    return pd.DataFrame(history)


def clear_history():

    history.clear()

    return pd.DataFrame(
        columns=[
            "Time",
            "Mode",
            "Gen-Z Score",
            "Tone"
        ]
    )


css = """

body {
    background: #08090d !important;
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    background: #08090d !important;
}

#hero {
    padding: 35px;
    border-radius: 25px;
    text-align: center;
    background: linear-gradient(
        135deg,
        #111827,
        #201044,
        #35113d
    );
    border: 1px solid #403052;
    margin-bottom: 20px;
}

#hero h1 {
    font-size: 48px !important;
    font-weight: 900 !important;
    margin-bottom: 5px;
}

#hero p {
    color: #b9b9c7;
    font-size: 17px;
}

.card {
    background: #11131a !important;
    border: 1px solid #292b36 !important;
    border-radius: 20px !important;
    padding: 20px !important;
}

#translation {
    font-size: 22px !important;
    font-weight: 600 !important;
}

button {
    border-radius: 14px !important;
}

textarea {
    border-radius: 14px !important;
}

"""


# ============================================================
# UI
# ============================================================

with gr.Blocks(
    title="🧃 Gen-Z Translator",
    theme=gr.themes.Base(
        primary_hue="purple",
        neutral_hue="slate"
    ),
    css=css
) as app:


    gr.HTML("""
    <div id="hero">

        <h1>🧃 GEN-Z TRANSLATOR</h1>

        <p>
        Normal English → Internet English
        </p>

        <p>
        Because sometimes "okay" needs to become
        <b>"bet 😭"</b>
        </p>

    </div>
    """)

    gr.Markdown("## 💬 Enter Your Message")

    with gr.Row():

        with gr.Column(
            scale=2,
            elem_classes="card"
        ):

            input_text = gr.Textbox(
                label="Your Text",
                placeholder=
                "Example: I am very tired after college today...",
                lines=8
            )

            with gr.Row():

                translate_btn = gr.Button(
                    "🧃 TRANSLATE",
                    variant="primary"
                )

                random_btn = gr.Button(
                    "🎲 RANDOM EXAMPLE"
                )

        with gr.Column(
            scale=1,
            elem_classes="card"
        ):

            mode = gr.Radio(
                choices=[
                    "Gen-Z Translation",
                    "Formal Translation",
                    "Funny Gen-Z"
                ],
                value="Gen-Z Translation",
                label="Translation Mode"
            )

            gr.Markdown("""
            ### Modes

            🧃 **Gen-Z**
            Normal → Gen-Z

            🎓 **Formal**
            Gen-Z → Professional

            😂 **Funny**
            Maximum internet energy
            """)

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    gr.Markdown("---")
    gr.Markdown("## ✨ Translation Result")

    output_text = gr.Textbox(
        label="Translated Message",
        lines=5,
        elem_id="translation"
    )


    gr.Markdown("## 📊 Language Analytics")

    with gr.Row():

        score = gr.Textbox(
            label="🧃 Gen-Z Score"
        )

        tone = gr.Textbox(
            label="🎭 Detected Tone"
        )

        slang = gr.Textbox(
            label="🔥 Slang Detected"
        )

        emojis = gr.Textbox(
            label="✨ Emoji Suggestions"
        )

    verdict = gr.Textbox(
        label="🤖 AI Verdict",
        lines=2
    )

    gr.Markdown("---")
    gr.Markdown("## ⚡ Try These")

    gr.Examples(
        examples=[
            ["Hello everyone, I am very excited about our project."],
            ["I am really tired after college today."],
            ["I cannot believe you actually did that!"],
            ["That was an amazing performance."],
            ["I do not know what is happening right now."],
            ["Thank you for helping me with this."]
        ],
        inputs=input_text
    )

    gr.Markdown("---")
    gr.Markdown("## 📈 Translation History")

    history_table = gr.Dataframe(
        headers=[
            "Time",
            "Mode",
            "Gen-Z Score",
            "Tone"
        ],
        interactive=False
    )

    with gr.Row():

        history_btn = gr.Button(
            "📊 Show History"
        )

        clear_btn = gr.Button(
            "🗑️ Clear History"
        )

    gr.HTML("""
    <div style="
        text-align:center;
        padding:30px;
        color:#666;
    ">
        🧃 Gen-Z Translator • Text & Speech Analysis Mini Project<br>
        <small>
        "No cap, this project is lowkey valid." 💀
        </small>
    </div>
    """)

    translate_btn.click(
        fn=analyze,
        inputs=[
            input_text,
            mode
        ],
        outputs=[
            output_text,
            score,
            tone,
            slang,
            emojis,
            verdict
        ]
    )

    random_btn.click(
        fn=random_example,
        inputs=None,
        outputs=input_text
    )

    history_btn.click(
        fn=show_history,
        inputs=None,
        outputs=history_table
    )

    clear_btn.click(
        fn=clear_history,
        inputs=None,
        outputs=history_table
    )


app.launch(
    share=True,
    debug=False
)

/tmp/ipykernel_1516/3123599651.py:495: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0e0c6bb70bc6ae2f3e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
